# 載入套件與資料準備
這裡將基礎套件載入，並準備好訓練資料。你可以在這裡切換你想測試的 Group。


In [1]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor


DATA_DIR = r"C:\Users\User\Desktop\高三專題\數據\ML\data"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"目前使用的裝置: {DEVICE}")

def load_and_split_group(group_prefixes):
    X_list, y_list = [], []
    for prefix in group_prefixes:
        X_path = os.path.join(DATA_DIR, f"{prefix}.X.npy")
        y_path = os.path.join(DATA_DIR, f"{prefix}.y.npy")
        if os.path.exists(X_path) and os.path.exists(y_path):
            X_list.append(np.load(X_path))
            y_list.append(np.load(y_path))
        else:
            print(f"Warning: 找不到對應的資料檔案 {prefix}")

    if not X_list: return None, None, None, None

    X_all = np.concatenate(X_list, axis=0).astype(np.float32)
    y_all = np.concatenate(y_list, axis=0).astype(np.float32)

    return train_test_split(X_all, y_all, test_size=0.2, random_state=42, shuffle=True)





目前使用的裝置: cpu


# 共用訓練函數
為了讓每個模型都能用同樣的邏輯進行訓練，我們把訓練過程包裝成函數。
訓練過程會使用 `reduction='mean'` 穩定梯度，測試時則使用 `reduction='sum'` 以計算總誤差。


In [2]:
def train_and_evaluate(model, X_train, y_train, X_test, y_test, epochs, lr, batch_size):
    model = model.to(DEVICE)
    train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
    test_dataset = TensorDataset(torch.tensor(X_test), torch.tensor(y_test))
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    # 訓練用 mean 穩定更新，測試用 sum 符合你的距離加總需求
    train_criterion = nn.L1Loss(reduction='mean')
    eval_criterion = nn.L1Loss(reduction='sum')
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    model.train()
    for epoch in range(epochs):
        total_train_loss = 0.0
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = train_criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            total_train_loss += eval_criterion(outputs, batch_y).item()
            
        # 每 5 個 epoch 或是開頭/結尾時印出 Loss
        if epoch == 0 or epoch == epochs - 1 or (epoch + 1) % 5 == 0:
            print(f"    [Epoch {epoch+1:02d}/{epochs}] Train Loss (Sum L1): {total_train_loss:.4f}")
            
    model.eval()
    total_test_loss = 0.0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
            outputs = model(batch_X)
            loss = eval_criterion(outputs, batch_y)
            total_test_loss += loss.item()
            
    print(f"  => 最終測試集總 L1 誤差: {total_test_loss:.4f}")
    return total_test_loss



## 1. Linear Regression


In [3]:


class LinearModel(nn.Module):
    def __init__(self, input_dim=50*24, output_dim=2):
        super().__init__()
        self.fc = nn.Linear(input_dim, output_dim)
        
    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.fc(x)



In [6]:
groups = {
    "Group 1 (ba, bc)": ["ba", "bc"],
    "Group 2 (ra, rc)": ["ra", "rc"],
    "Group 3 (s1, s2)": ["s1", "s2"]
}

EPOCHS_LINEAR = 200
LR_LINEAR = 1e-4
BATCH_SIZE_LINEAR = 128


for group_name, prefixes in groups.items():
    print(f"\n========== 開始訓練 {group_name} ==========")
    
    # 1. 載入並切割該組別的資料
    X_train_g, X_test_g, y_train_g, y_test_g = load_and_split_group(prefixes)
    
    if X_train_g is not None:
        # 2. ⚠️ 非常重要：每次迴圈都要「重新宣告」一個全新乾淨的模型！
        model_linear_g = LinearModel()
        
        # 3. 將資料與模型送進你前面寫好的 train_and_evaluate 函數
        train_and_evaluate(
            model=model_linear_g, 
            X_train=X_train_g, 
            y_train=y_train_g, 
            X_test=X_test_g, 
            y_test=y_test_g, 
            epochs=EPOCHS_LINEAR, 
            lr=LR_LINEAR, 
            batch_size=BATCH_SIZE_LINEAR
        )



========== 開始訓練 Group 1 (ba, bc) ==========
    [Epoch 01/200] Train Loss (Sum L1): 1842.8717
    [Epoch 05/200] Train Loss (Sum L1): 967.4265
    [Epoch 10/200] Train Loss (Sum L1): 721.2483
    [Epoch 15/200] Train Loss (Sum L1): 598.3018
    [Epoch 20/200] Train Loss (Sum L1): 532.5069
    [Epoch 25/200] Train Loss (Sum L1): 493.0811
    [Epoch 30/200] Train Loss (Sum L1): 469.6542
    [Epoch 35/200] Train Loss (Sum L1): 457.4012
    [Epoch 40/200] Train Loss (Sum L1): 441.4902
    [Epoch 45/200] Train Loss (Sum L1): 435.5191
    [Epoch 50/200] Train Loss (Sum L1): 428.0563
    [Epoch 55/200] Train Loss (Sum L1): 422.6203
    [Epoch 60/200] Train Loss (Sum L1): 420.7816
    [Epoch 65/200] Train Loss (Sum L1): 416.3489
    [Epoch 70/200] Train Loss (Sum L1): 412.7953
    [Epoch 75/200] Train Loss (Sum L1): 410.7903
    [Epoch 80/200] Train Loss (Sum L1): 411.8609
    [Epoch 85/200] Train Loss (Sum L1): 407.4555
    [Epoch 90/200] Train Loss (Sum L1): 405.6211
    [Epoch 95/200] Trai

In [6]:
# 1. 專屬的讀取函數 (抓取 _linear.npy 檔案)
def load_and_split_group_linear(group_prefixes):
    X_list, y_list = [], []
    for prefix in group_prefixes:
        # 注意這裡讀取的是剛剛產生的 _linear 檔案
        X_path = os.path.join(DATA_DIR, f"{prefix}.X_linear.npy")
        y_path = os.path.join(DATA_DIR, f"{prefix}.y_linear.npy")
        if os.path.exists(X_path) and os.path.exists(y_path):
            X_list.append(np.load(X_path))
            y_list.append(np.load(y_path))
            
    if not X_list: return None, None, None, None
    X_all = np.concatenate(X_list, axis=0).astype(np.float32)
    y_all = np.concatenate(y_list, axis=0).astype(np.float32)
    
    return train_test_split(X_all, y_all, test_size=0.2, random_state=42, shuffle=True)
# 2. 定義針對「瞬間 24 個特徵」的線性模型
class InstantLinearModel(nn.Module):
    def __init__(self, input_dim=24, output_dim=2):  # 輸入維度從 1200 變成 24
        super().__init__()
        self.fc = nn.Linear(input_dim, output_dim)
        
    def forward(self, x):
        # x 已經是乾淨的 (batch_size, 24) 了，不需要攤平，直接進入全連接層
        return self.fc(x)
# ==== 開始訓練 ====
groups = {
    "Group 1 (ba, bc)": ["ba", "bc"],
    "Group 2 (ra, rc)": ["ra", "rc"],
    "Group 3 (s1, s2)": ["s1", "s2"]
}
# 參數設定
EPOCHS_INSTANT_LIN = 200
LR_INSTANT_LIN = 1e-4
BATCH_SIZE_INSTANT_LIN = 32
for group_name, prefixes in groups.items():
    print(f"\n========== 瞬間線性模型 (Instant Linear): {group_name} ==========")
    
    # 呼叫新的線性資料讀取函數
    X_train_lin, X_test_lin, y_train_lin, y_test_lin = load_and_split_group_linear(prefixes)
    
    if X_train_lin is not None:
        # 建立全新模型
        model_instant_lin = InstantLinearModel()
        
        # 直接使用你原本寫好的 train_and_evaluate！
        train_and_evaluate(
            model=model_instant_lin, 
            X_train=X_train_lin, 
            y_train=y_train_lin, 
            X_test=X_test_lin, 
            y_test=y_test_lin, 
            epochs=EPOCHS_INSTANT_LIN, 
            lr=LR_INSTANT_LIN, 
            batch_size=BATCH_SIZE_INSTANT_LIN
        )


========== 瞬間線性模型 (Instant Linear): Group 1 (ba, bc) ==========
    [Epoch 01/200] Train Loss (Sum L1): 2295.9857
    [Epoch 05/200] Train Loss (Sum L1): 1754.8322
    [Epoch 10/200] Train Loss (Sum L1): 1305.6417
    [Epoch 15/200] Train Loss (Sum L1): 1017.2679
    [Epoch 20/200] Train Loss (Sum L1): 833.0096
    [Epoch 25/200] Train Loss (Sum L1): 724.1538
    [Epoch 30/200] Train Loss (Sum L1): 663.1002
    [Epoch 35/200] Train Loss (Sum L1): 629.4763
    [Epoch 40/200] Train Loss (Sum L1): 610.9505
    [Epoch 45/200] Train Loss (Sum L1): 601.1198
    [Epoch 50/200] Train Loss (Sum L1): 595.6416
    [Epoch 55/200] Train Loss (Sum L1): 592.3844
    [Epoch 60/200] Train Loss (Sum L1): 590.5454
    [Epoch 65/200] Train Loss (Sum L1): 589.3666
    [Epoch 70/200] Train Loss (Sum L1): 588.4724
    [Epoch 75/200] Train Loss (Sum L1): 588.1428
    [Epoch 80/200] Train Loss (Sum L1): 587.7518
    [Epoch 85/200] Train Loss (Sum L1): 587.7212
    [Epoch 90/200] Train Loss (Sum L1): 587.5142


## 2. Neural Network


In [11]:
class NNModel(nn.Module):
    def __init__(self, input_dim=50*24, output_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, output_dim)
        )
        
    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.net(x)



In [24]:
groups = {
    "Group 1 (ba, bc)": ["ba", "bc"],
    "Group 2 (ra, rc)": ["ra", "rc"],
    "Group 3 (s1, s2)": ["s1", "s2"]
}

EPOCHS_NN = 100
LR_NN = 1e-4
BATCH_SIZE_NN = 32


for group_name, prefixes in groups.items():
    print(f"\n========== 開始訓練 {group_name} ==========")
    
    # 1. 載入並切割該組別的資料
    X_train_g, X_test_g, y_train_g, y_test_g = load_and_split_group(prefixes)
    
    if X_train_g is not None:
        # 2. ⚠️ 非常重要：每次迴圈都要「重新宣告」一個全新乾淨的模型！
        model_nn_g = NNModel()
        
        # 3. 將資料與模型送進你前面寫好的 train_and_evaluate 函數
        train_and_evaluate(
            model=model_nn_g, 
            X_train=X_train_g, 
            y_train=y_train_g, 
            X_test=X_test_g, 
            y_test=y_test_g, 
            epochs=EPOCHS_NN, 
            lr=LR_NN, 
            batch_size=BATCH_SIZE_NN
        )



========== 開始訓練 Group 1 (ba, bc) ==========
    [Epoch 01/100] Train Loss (Sum L1): 589.2869
    [Epoch 05/100] Train Loss (Sum L1): 357.8657
    [Epoch 10/100] Train Loss (Sum L1): 273.4014
    [Epoch 15/100] Train Loss (Sum L1): 219.4791
    [Epoch 20/100] Train Loss (Sum L1): 183.7764
    [Epoch 25/100] Train Loss (Sum L1): 159.6724
    [Epoch 30/100] Train Loss (Sum L1): 145.2716
    [Epoch 35/100] Train Loss (Sum L1): 125.9008
    [Epoch 40/100] Train Loss (Sum L1): 109.1672
    [Epoch 45/100] Train Loss (Sum L1): 98.3810
    [Epoch 50/100] Train Loss (Sum L1): 96.8923
    [Epoch 55/100] Train Loss (Sum L1): 90.6715
    [Epoch 60/100] Train Loss (Sum L1): 84.7179
    [Epoch 65/100] Train Loss (Sum L1): 78.8998
    [Epoch 70/100] Train Loss (Sum L1): 82.7531
    [Epoch 75/100] Train Loss (Sum L1): 73.3678
    [Epoch 80/100] Train Loss (Sum L1): 75.2653
    [Epoch 85/100] Train Loss (Sum L1): 71.6560
    [Epoch 90/100] Train Loss (Sum L1): 70.0001
    [Epoch 95/100] Train Loss (Sum

KeyboardInterrupt: 

## 3. CNN Model


In [18]:
# ==== 簡單版 CNN 模型與分組訓練程式碼 ====

class CNNModel(nn.Module):
    def __init__(self, num_features=24, output_dim=2):
        super().__init__()
        # PyTorch 的 Conv1d 預期輸入為 (batch, channels, length)
        # 我們的資料原本是 (batch, 50, 24)，所以在 forward 會先轉置
        self.conv = nn.Sequential(
            nn.Conv1d(in_channels=num_features, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2), # 長度 50 變成 25
            
            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2)  # 長度 25 變成 12
        )
        self.fc = nn.Sequential(
            nn.Linear(64 * 12, 64),
            nn.ReLU(),
            nn.Linear(64, output_dim)
        )

    def forward(self, x):
        # 步驟 1: 轉換維度，把「特徵(24)」放到中間，「時間(50)」移到最後面
        x = x.permute(0, 2, 1)    # 維度變成 (batch_size, 24, 50)
        
        # 步驟 2: 經過兩層卷積
        x = self.conv(x)
        
        # 步驟 3: 將特徵攤平
        x = x.view(x.size(0), -1) # 攤平成 (batch_size, 64*12)
        
        # 步驟 4: 經過全連接層輸出 2 個預測值
        return self.fc(x)



In [23]:
groups = {
    "Group 1 (ba, bc)": ["ba", "bc"],
    "Group 2 (ra, rc)": ["ra", "rc"],
    "Group 3 (s1, s2)": ["s1", "s2"]
}

EPOCHS_CNN = 100
LR_CNN = 1e-4
BATCH_SIZE_CNN = 32


for group_name, prefixes in groups.items():
    print(f"\n========== 開始訓練 {group_name} ==========")
    
    # 1. 載入並切割該組別的資料
    X_train_g, X_test_g, y_train_g, y_test_g = load_and_split_group(prefixes)
    
    if X_train_g is not None:
        # 2. ⚠️ 非常重要：每次迴圈都要「重新宣告」一個全新乾淨的模型！
        model_cnn_g = CNNModel()
        
        # 3. 將資料與模型送進你前面寫好的 train_and_evaluate 函數
        train_and_evaluate(
            model=model_cnn_g, 
            X_train=X_train_g, 
            y_train=y_train_g, 
            X_test=X_test_g, 
            y_test=y_test_g, 
            epochs=EPOCHS_CNN, 
            lr=LR_CNN, 
            batch_size=BATCH_SIZE_CNN
        )



========== 開始訓練 Group 1 (ba, bc) ==========
    [Epoch 01/100] Train Loss (Sum L1): 685.5064
    [Epoch 05/100] Train Loss (Sum L1): 474.2846
    [Epoch 10/100] Train Loss (Sum L1): 424.7022
    [Epoch 15/100] Train Loss (Sum L1): 395.0893
    [Epoch 20/100] Train Loss (Sum L1): 367.4823
    [Epoch 25/100] Train Loss (Sum L1): 344.4798
    [Epoch 30/100] Train Loss (Sum L1): 322.4175
    [Epoch 35/100] Train Loss (Sum L1): 306.1712
    [Epoch 40/100] Train Loss (Sum L1): 287.6278
    [Epoch 45/100] Train Loss (Sum L1): 273.9296
    [Epoch 50/100] Train Loss (Sum L1): 256.5398
    [Epoch 55/100] Train Loss (Sum L1): 241.1402
    [Epoch 60/100] Train Loss (Sum L1): 233.5712
    [Epoch 65/100] Train Loss (Sum L1): 219.6761
    [Epoch 70/100] Train Loss (Sum L1): 208.2451
    [Epoch 75/100] Train Loss (Sum L1): 199.2148
    [Epoch 80/100] Train Loss (Sum L1): 188.0547
    [Epoch 85/100] Train Loss (Sum L1): 177.9358
    [Epoch 90/100] Train Loss (Sum L1): 170.9092
    [Epoch 95/100] Train

KeyboardInterrupt: 

## 4. LSTM Model


In [4]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=24, hidden_size=64, output_dim=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_dim)

    def forward(self, x):
        out, (hn, cn) = self.lstm(x)
        last_out = out[:, -1, :] 
        return self.fc(last_out)



In [6]:
def train_lstm_custom_lr(model, X_train, y_train, X_test, y_test, epochs, batch_size):
    model = model.to(DEVICE)
    train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
    test_dataset = TensorDataset(torch.tensor(X_test), torch.tensor(y_test))
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    train_criterion = nn.L1Loss(reduction='mean')
    eval_criterion = nn.L1Loss(reduction='sum')
    
    # 🌟 初始學習率設為 1e-3
    current_lr = 1e-3
    optimizer = optim.Adam(model.parameters(), lr=current_lr)
    
    print(f"🚀 開始訓練，初始學習率: {current_lr}")
    model.train()
    for epoch in range(epochs):
        total_train_loss = 0.0
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = train_criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            total_train_loss += eval_criterion(outputs, batch_y).item()
            
        # 每 5 個 epoch 印出一次狀況
        if epoch == 0 or epoch == epochs - 1 or (epoch + 1) % 5 == 0:
            print(f"    [Epoch {epoch+1:03d}/{epochs}] Train Loss: {total_train_loss:.4f} | LR: {current_lr}")
            
        # 🌟 核心邏輯：當 Loss 降到 50 以下，且學習率還沒降過時，進行降檔！
        if total_train_loss < 50.0 and current_lr > 1e-4:
            current_lr = 1e-4
            for param_group in optimizer.param_groups:
                param_group['lr'] = current_lr
            print(f"    🔥 [觸發微調] Train Loss 降至 {total_train_loss:.2f} < 50！")
            print(f"    ⚙️ 學習率已切換為更細緻的 {current_lr} 進行收斂！")
            
    # ====== 測試階段 ======
    model.eval()
    total_test_loss = 0.0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
            outputs = model(batch_X)
            loss = eval_criterion(outputs, batch_y)
            total_test_loss += loss.item()
            
    print(f"  => 🏆 最終測試集 L1 總誤差: {total_test_loss:.4f}")
    return total_test_loss


In [25]:
groups = {
    "Group 1 (ba, bc)": ["ba", "bc"],
    "Group 2 (ra, rc)": ["ra", "rc"],
    "Group 3 (s1, s2)": ["s1", "s2"]
}

EPOCHS_LSTM = 100
LR_LSTM = 1e-4
BATCH_SIZE_LSTM = 32


for group_name, prefixes in groups.items():
    print(f"\n========== 開始訓練 {group_name} ==========")
    
    # 1. 載入並切割該組別的資料
    X_train_g, X_test_g, y_train_g, y_test_g = load_and_split_group(prefixes)
    
    if X_train_g is not None:
        # 2. ⚠️ 非常重要：每次迴圈都要「重新宣告」一個全新乾淨的模型！
        model_lstm_g = LSTMModel()
        
        # 3. 將資料與模型送進你前面寫好的 train_and_evaluate 函數
        train_and_evaluate(
            model=model_lstm_g, 
            X_train=X_train_g, 
            y_train=y_train_g, 
            X_test=X_test_g, 
            y_test=y_test_g, 
            epochs=EPOCHS_LSTM, 
            lr=LR_LSTM, 
            batch_size=BATCH_SIZE_LSTM
        )



========== 開始訓練 Group 1 (ba, bc) ==========
    [Epoch 01/100] Train Loss (Sum L1): 791.6348
    [Epoch 05/100] Train Loss (Sum L1): 543.3461
    [Epoch 10/100] Train Loss (Sum L1): 462.3866
    [Epoch 15/100] Train Loss (Sum L1): 414.6414
    [Epoch 20/100] Train Loss (Sum L1): 389.2864
    [Epoch 25/100] Train Loss (Sum L1): 371.5668
    [Epoch 30/100] Train Loss (Sum L1): 357.7484
    [Epoch 35/100] Train Loss (Sum L1): 345.0243
    [Epoch 40/100] Train Loss (Sum L1): 334.0368
    [Epoch 45/100] Train Loss (Sum L1): 325.8845
    [Epoch 50/100] Train Loss (Sum L1): 317.0765
    [Epoch 55/100] Train Loss (Sum L1): 308.8958
    [Epoch 60/100] Train Loss (Sum L1): 303.1501
    [Epoch 65/100] Train Loss (Sum L1): 297.2525
    [Epoch 70/100] Train Loss (Sum L1): 291.8259
    [Epoch 75/100] Train Loss (Sum L1): 286.4951
    [Epoch 80/100] Train Loss (Sum L1): 282.1439
    [Epoch 85/100] Train Loss (Sum L1): 277.3411
    [Epoch 90/100] Train Loss (Sum L1): 273.5178
    [Epoch 95/100] Train

KeyboardInterrupt: 

In [22]:
# 調lr

groups = {
    "Group 1 (ba, bc)": ["ba", "bc"],
    "Group 2 (ra, rc)": ["ra", "rc"],
    "Group 3 (s1, s2)": ["s1", "s2"]
}

EPOCHS_LSTM = 100
BATCH_SIZE_LSTM = 32


for group_name, prefixes in groups.items():
    print(f"\n========== 開始訓練 {group_name} ==========")
    
    # 1. 載入並切割該組別的資料
    X_train_g, X_test_g, y_train_g, y_test_g = load_and_split_group(prefixes)
    
    if X_train_g is not None:
        # 2. ⚠️ 非常重要：每次迴圈都要「重新宣告」一個全新乾淨的模型！
        model_lstm_g = LSTMModel()
        
        # 3. 將資料與模型送進你前面寫好的 train_and_evaluate 函數
        train_lstm_custom_lr(
            model=model_lstm_g, 
            X_train=X_train_g, 
            y_train=y_train_g, 
            X_test=X_test_g, 
            y_test=y_test_g, 
            epochs=EPOCHS_LSTM,  
            batch_size=BATCH_SIZE_LSTM
        )



========== 開始訓練 Group 1 (ba, bc) ==========
🚀 開始訓練，初始學習率: 0.001
    [Epoch 001/100] Train Loss: 589.5013 | LR: 0.001
    [Epoch 005/100] Train Loss: 371.4725 | LR: 0.001
    [Epoch 010/100] Train Loss: 314.0916 | LR: 0.001
    [Epoch 015/100] Train Loss: 280.0816 | LR: 0.001
    [Epoch 020/100] Train Loss: 253.2100 | LR: 0.001
    [Epoch 025/100] Train Loss: 234.9801 | LR: 0.001
    [Epoch 030/100] Train Loss: 222.4677 | LR: 0.001
    [Epoch 035/100] Train Loss: 212.7715 | LR: 0.001
    [Epoch 040/100] Train Loss: 193.4952 | LR: 0.001
    [Epoch 045/100] Train Loss: 186.1706 | LR: 0.001
    [Epoch 050/100] Train Loss: 172.8822 | LR: 0.001
    [Epoch 055/100] Train Loss: 166.1926 | LR: 0.001
    [Epoch 060/100] Train Loss: 157.2477 | LR: 0.001
    [Epoch 065/100] Train Loss: 151.9189 | LR: 0.001
    [Epoch 070/100] Train Loss: 145.5592 | LR: 0.001
    [Epoch 075/100] Train Loss: 136.8972 | LR: 0.001
    [Epoch 080/100] Train Loss: 132.7551 | LR: 0.001
    [Epoch 085/100] Train Loss: 12

KeyboardInterrupt: 

In [7]:
# train set包含test set

def load_and_cheat_split_group(group_prefixes):
    """
    作弊版資料載入 (可直接比較版)：
    訓練集為 100% 全部的資料 (作弊看答案)
    測試集為原本固定切出來的 20% 資料 (讓總和數字可以跟以前比較)
    """
    X_list, y_list = [], []
    for prefix in group_prefixes:
        X_path = os.path.join(DATA_DIR, f"{prefix}.X.npy")
        y_path = os.path.join(DATA_DIR, f"{prefix}.y.npy")
        if os.path.exists(X_path) and os.path.exists(y_path):
            X_list.append(np.load(X_path))
            y_list.append(np.load(y_path))
            
    if not X_list: return None, None, None, None
    
    X_all = np.concatenate(X_list, axis=0).astype(np.float32)
    y_all = np.concatenate(y_list, axis=0).astype(np.float32)
    
    # 正常切分，抽出那 1/5 的測試集 (random_state=42 確保題目跟以前一模一樣)
    _, X_test, _, y_test = train_test_split(X_all, y_all, test_size=0.2, random_state=42, shuffle=True)
    
    # ⚠️ 關鍵作弊點：訓練時我們不給切過的 80%，我們直接給它 100% 的 X_all！
    return X_all, X_test, y_all, y_test

# ==== 開始進行作弊測試 ====

groups = {
    "Group 1 (ba, bc)": ["ba", "bc"],
    "Group 2 (ra, rc)": ["ra", "rc"],
    "Group 3 (s1, s2)": ["s1", "s2"]
}

EPOCHS_CHEAT = 750  # 讓它多背幾次
LR_CHEAT = 1e-3

for group_name, prefixes in groups.items():
    print(f"\n========== 作弊極限測試 (1/5 Test Set): {group_name} ==========")
    
    # 1. 呼叫我們剛剛寫的可比較作弊版載入函數
    X_train_cheat, X_test_cheat, y_train_cheat, y_test_cheat = load_and_cheat_split_group(prefixes)
    
    if X_train_cheat is not None:
        model_lstm_cheat = LSTMModel()
        
        train_and_evaluate(
            model=model_lstm_cheat, 
            X_train=X_train_cheat, 
            y_train=y_train_cheat, 
            X_test=X_test_cheat, 
            y_test=y_test_cheat, 
            epochs=EPOCHS_CHEAT, 
            lr=LR_CHEAT, 
            batch_size=32
        )



========== 作弊極限測試 (1/5 Test Set): Group 1 (ba, bc) ==========
    [Epoch 01/750] Train Loss (Sum L1): 723.9221
    [Epoch 05/750] Train Loss (Sum L1): 433.8870
    [Epoch 10/750] Train Loss (Sum L1): 364.5881
    [Epoch 15/750] Train Loss (Sum L1): 334.9960
    [Epoch 20/750] Train Loss (Sum L1): 304.9293
    [Epoch 25/750] Train Loss (Sum L1): 287.2881
    [Epoch 30/750] Train Loss (Sum L1): 270.5103
    [Epoch 35/750] Train Loss (Sum L1): 255.0750
    [Epoch 40/750] Train Loss (Sum L1): 240.5593
    [Epoch 45/750] Train Loss (Sum L1): 227.8338
    [Epoch 50/750] Train Loss (Sum L1): 220.0751
    [Epoch 55/750] Train Loss (Sum L1): 209.1004
    [Epoch 60/750] Train Loss (Sum L1): 199.1361
    [Epoch 65/750] Train Loss (Sum L1): 189.8847
    [Epoch 70/750] Train Loss (Sum L1): 181.7045
    [Epoch 75/750] Train Loss (Sum L1): 175.7463
    [Epoch 80/750] Train Loss (Sum L1): 167.7396
    [Epoch 85/750] Train Loss (Sum L1): 160.5624
    [Epoch 90/750] Train Loss (Sum L1): 155.6143
    [E

In [ ]:
# 互相預測

# 定義簡化版的名稱方便表格顯示
groups_cross = {
    "Bend": ["ba", "bc"],
    "RightAngle": ["ra", "rc"],
    "Straight": ["s1", "s2"]
}

# 參數設定
EPOCHS_CROSS = 300
LR_CROSS = 1e-3
BATCH_SIZE_CROSS = 32

# ==== 1. 拆解訓練與測試函數 ====
def train_only(model, X_train, y_train, epochs, lr, batch_size):
    """純訓練函數：只負責訓練並回傳訓練好的模型"""
    model = model.to(DEVICE)
    train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    criterion = nn.L1Loss(reduction='mean')
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    model.train()
    for epoch in range(epochs):
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
    return model

def evaluate_only(model, X_test, y_test, batch_size=32):
    """純測試函數：負責給定測試卷並回傳 L1 總誤差"""
    model.eval()
    test_dataset = TensorDataset(torch.tensor(X_test), torch.tensor(y_test))
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    criterion = nn.L1Loss(reduction='sum')
    
    total_loss = 0.0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
            outputs = model(batch_X)
            total_loss += criterion(outputs, batch_y).item()
            
    return total_loss

# ==== 2. 預先載入所有資料 ====
print("📦 正在載入所有資料...")
loaded_data = {}
for name, prefixes in groups_cross.items():
    X_tr, X_te, y_tr, y_te = load_and_split_group(prefixes)
    loaded_data[name] = {
        'X_train': X_tr, 'y_train': y_tr,
        'X_test': X_te, 'y_test': y_te
    }

# ==== 3. 進行 3x3 交叉訓練與測試 ====
# matrix[train_name][test_name] = loss
results_matrix = {t: {} for t in groups_cross.keys()}

for train_name in groups_cross.keys():
    print(f"\n🚀 正在訓練專屬模型: [ {train_name} 專家 ] ... (請稍候)")
    X_tr = loaded_data[train_name]['X_train']
    y_tr = loaded_data[train_name]['y_train']
    
    # 初始化一個全新乾淨的 LSTM
    model = LSTMModel()
    
    # 訓練模型
    trained_model = train_only(model, X_tr, y_tr, epochs=EPOCHS_CROSS, lr=LR_CROSS, batch_size=BATCH_SIZE_CROSS)
    
    # 考 3 張考卷
    for test_name in groups_cross.keys():
        X_te = loaded_data[test_name]['X_test']
        y_te = loaded_data[test_name]['y_test']
        
        # 進行預測
        loss = evaluate_only(trained_model, X_te, y_te)
        results_matrix[train_name][test_name] = round(loss, 2)
        print(f"  => 拿 [ {train_name} 專家 ] 去考 [ {test_name} ] 的考卷: 誤差 {loss:.2f}")

# ==== 4. 畫出精美的 Pandas 矩陣表 ====
print("\n" + "="*60)
print("🏆 LSTM 跨步態預測交叉矩陣表 (數值為總 L1 誤差，越小越好)")
print("="*60)



📦 正在載入所有資料...

🚀 正在訓練專屬模型: [ Bend 專家 ] ... (請稍候)
  => 拿 [ Bend 專家 ] 去考 [ Bend ] 的考卷: 誤差 80.77
  => 拿 [ Bend 專家 ] 去考 [ RightAngle ] 的考卷: 誤差 82.41
  => 拿 [ Bend 專家 ] 去考 [ Straight ] 的考卷: 誤差 204.11

🚀 正在訓練專屬模型: [ RightAngle 專家 ] ... (請稍候)
  => 拿 [ RightAngle 專家 ] 去考 [ Bend ] 的考卷: 誤差 90.31
  => 拿 [ RightAngle 專家 ] 去考 [ RightAngle ] 的考卷: 誤差 77.51
  => 拿 [ RightAngle 專家 ] 去考 [ Straight ] 的考卷: 誤差 173.26

🚀 正在訓練專屬模型: [ Straight 專家 ] ... (請稍候)
  => 拿 [ Straight 專家 ] 去考 [ Bend ] 的考卷: 誤差 197.31
  => 拿 [ Straight 專家 ] 去考 [ RightAngle ] 的考卷: 誤差 189.05
  => 拿 [ Straight 專家 ] 去考 [ Straight ] 的考卷: 誤差 65.28

🏆 LSTM 跨步態預測交叉矩陣表 (數值為總 L1 誤差，越小越好)


NameError: name 'pd' is not defined

In [13]:
# 把字典轉換成 DataFrame 顯示
df_results = pd.DataFrame(results_matrix).T
df_results.index.name = '訓練資料 (Train)'
df_results.columns.name = '測試資料 (Test)'

print(df_results)


測試資料 (Test)     Bend  RightAngle  Straight
訓練資料 (Train)                              
Bend           80.77       82.41    204.11
RightAngle     90.31       77.51    173.26
Straight      197.31      189.05     65.28


## 5. GBRT

In [59]:
# 👉 在這裡獨立調整 GBRT 的參數
MAX_ITER_GBRT = 200   # 相當於 Epochs，通常設 100~300
LR_GBRT = 0.1    # GBRT 的學習率通常比神經網路大 (例如 0.1)

In [60]:
groups = {
    "Group 1 (ba, bc)": ["ba", "bc"],
    "Group 2 (ra, rc)": ["ra", "rc"],
    "Group 3 (s1, s2)": ["s1", "s2"]
}

for group_name, prefixes in groups.items():
    print(f"\n========== 開始訓練 {group_name} ==========")
    
    # 1. 載入並切割該組別的資料
    X_train_g, X_test_g, y_train_g, y_test_g = load_and_split_group(prefixes)
    
    if X_train_g is not None:
        # 2. GBRT 吃的是 2D 矩陣，所以這裡必須把 (batch, 50, 24) 攤平成 (batch, 1200)
        X_train_flat = X_train_g.reshape(X_train_g.shape[0], -1)
        X_test_flat = X_test_g.reshape(X_test_g.shape[0], -1)
        
        # 3. ⚠️ 非常重要：每次迴圈都要「重新宣告」一個全新乾淨的模型！
        base_gbdt = HistGradientBoostingRegressor(
            loss='absolute_error', 
            max_iter=MAX_ITER_GBRT,
            learning_rate=LR_GBRT,
            random_state=42
        )
        model_gbdt_g = MultiOutputRegressor(base_gbdt)
        
        # 4. 訓練與評估 (因為它是 sklearn 套件，不用呼叫之前的 train_and_evaluate)
        print("  正在訓練中 (這不會顯示進度條，請耐心等幾秒鐘)...")
        model_gbdt_g.fit(X_train_flat, y_train_g)
        
        # 測試集預測
        y_pred = model_gbdt_g.predict(X_test_flat)
        
        # 計算測試集的總 L1 誤差 (每一項相減取絕對值後加總)
        total_test_loss = np.sum(np.abs(y_pred - y_test_g))
        print(f"👉 {group_name} [GBRT] 最終測試集 L1 總誤差: {total_test_loss:.4f}")



========== 開始訓練 Group 1 (ba, bc) ==========
  正在訓練中 (這不會顯示進度條，請耐心等幾秒鐘)...
👉 Group 1 (ba, bc) [GBRT] 最終測試集 L1 總誤差: 87.3077

========== 開始訓練 Group 2 (ra, rc) ==========
  正在訓練中 (這不會顯示進度條，請耐心等幾秒鐘)...


KeyboardInterrupt: 

## 6.test

In [61]:
groups = {
    "Group 1 (ba, bc)": ["ba", "bc"],
    "Group 2 (ra, rc)": ["ra", "rc"],
    "Group 3 (s1, s2)": ["s1", "s2"]
}

print("========== 基準線測試 (Baseline Test) ==========")

for group_name, prefixes in groups.items():
    print(f"\n--- 測試組別: {group_name} ---")
    
    # 載入該組別資料
    X_train_g, X_test_g, y_train_g, y_test_g = load_and_split_group(prefixes)
    
    if X_train_g is not None:
        # 1. 常數零預測 (全部猜 0)
        y_pred_zero = np.zeros_like(y_test_g)
        total_zero_loss = np.sum(np.abs(y_pred_zero - y_test_g))
        
        # 2. 訓練集平均值預測 (全部猜歷史平均)
        mean_y = y_train_g.mean(axis=0)
        y_pred_mean = np.full_like(y_test_g, mean_y)
        total_mean_loss = np.sum(np.abs(y_pred_mean - y_test_g))
        
        print(f"👉 【全部無腦猜 0】 L1 總誤差: {total_zero_loss:.4f}")
        print(f"👉 【全部猜平均值】 L1 總誤差: {total_mean_loss:.4f}")


========== 基準線測試 (Baseline Test) ==========

--- 測試組別: Group 1 (ba, bc) ---
👉 【全部無腦猜 0】 L1 總誤差: 211.6550
👉 【全部猜平均值】 L1 總誤差: 212.3559

--- 測試組別: Group 2 (ra, rc) ---
👉 【全部無腦猜 0】 L1 總誤差: 197.2340
👉 【全部猜平均值】 L1 總誤差: 197.2751

--- 測試組別: Group 3 (s1, s2) ---
👉 【全部無腦猜 0】 L1 總誤差: 221.1410
👉 【全部猜平均值】 L1 總誤差: 217.8840
